In [2]:
import pandas as pd
import numpy as np

## Detecting and Filtering Outliers

Outlier handling is done with standard array operations — boolean indexing and masking.

### Finding Outliers

- `col.abs() > threshold` → boolean mask of outliers in one column.
- `(df.abs() > threshold).any(axis="columns")` → boolean mask of **rows** containing at least one outlier.
  - Parentheses around `df.abs() > threshold` are required before calling `.any()`.

### Capping Outliers

Use boolean indexing on the left-hand side of an assignment to replace outlier values:

```python
data[data.abs() > 3] = np.sign(data) * 3
```

- `np.sign(data)` returns `+1` for positive values and `-1` for negative values.
- Multiplying by the threshold caps each outlier at `±3` while preserving its sign.

## Key Points

- Use `.abs()` with a threshold to create a boolean outlier mask.
- `.any(axis="columns")` flags rows where **any** column is an outlier.
- Assign to a masked selection to cap or replace outlier values in place.
- `np.sign(x)` returns `+1` / `-1` — useful for sign-preserving capping.

In [3]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.007741,0.079308,0.003548,-0.067147
std,0.979298,0.991918,1.068789,1.001118
min,-3.170244,-3.138162,-4.408290,-3.055118
25%,-0.662824,-0.633328,-0.694559,-0.742919
50%,-0.011946,0.096389,-0.022947,-0.071891
75%,0.624860,0.744242,0.649496,0.552247
max,3.769388,3.505722,3.874051,3.014031


In [4]:
# Values in column 2 with absolute value > 3
col = data[2]

col[col.abs() > 3]

44     3.505134
71     3.030669
75    -4.408290
134    3.272266
208    3.276426
417   -3.693427
462   -3.280792
550    3.471691
679    3.106781
767    3.874051
832   -3.392624
849   -3.075409
867   -3.539799
Name: 2, dtype: float64

In [5]:
# All rows containing at least one outlier
data[(data.abs() > 3).any(axis="columns")]

,0,1,2,3
44,1.251565,0.827333,3.505134,-1.928700
71,-0.125383,-0.467868,3.030669,0.163411
75,-0.731005,1.690813,-4.408290,0.243962
79,3.372005,1.483414,1.388739,0.161601
134,-0.665346,-0.605690,3.272266,1.014468
169,0.513797,0.403940,-1.109915,3.014031
208,0.288655,-0.564900,3.276426,-0.588036
229,-0.760798,1.138933,-0.793681,-3.055118
279,-3.170244,-0.084289,-1.482438,-1.184710
417,1.598337,-0.293777,-3.693427,0.397926


In [6]:
# Cap all values outside [-3, 3] at ±3
data[data.abs() > 3] = np.sign(data) * 3

data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.008712,0.078747,0.004402,-0.067099
std,0.974869,0.989265,1.049905,1.000892
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.662824,-0.633328,-0.694559,-0.742919
50%,-0.011946,0.096389,-0.022947,-0.071891
75%,0.624860,0.744242,0.649496,0.552247
max,3.000000,3.000000,3.000000,3.000000


In [7]:
np.sign(data).head()  # +1 for positive, -1 for negative

,0,1,2,3
0,1.0,1.0,-1.0,1.0
1,-1.0,1.0,1.0,-1.0
2,1.0,-1.0,1.0,1.0
3,1.0,1.0,-1.0,-1.0
4,-1.0,-1.0,1.0,-1.0


## Permutation and Random Sampling

### Permutation — Reorder All Rows or Columns

`np.random.permutation(n)` generates a random integer ordering of length `n`.

Use it with:

- `df.take(sampler)` — select rows by integer position in the given order.
- `df.iloc[sampler]` — equivalent iloc-based approach.
- `df.take(sampler, axis="columns")` — permute columns instead of rows.

### Random Sampling — Select a Subset

`df.sample(n=k)` selects `k` rows at random **without replacement** (no row repeated).

Pass `replace=True` to sample **with replacement** (rows can repeat) — useful for bootstrapping.

## Key Points

- `np.random.permutation(n)` → full random reordering; use with `take` or `iloc`.
- `take(sampler, axis="columns")` permutes columns.
- `sample(n=k)` → random subset without replacement.
- `sample(n=k, replace=True)` → random subset with replacement (rows can repeat).

In [8]:
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))

df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [9]:
sampler = np.random.permutation(5)

sampler

array([2, 0, 4, 3, 1])

In [10]:
df.take(sampler)  # reorder rows

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13


In [11]:
df.iloc[sampler]  # equivalent

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13


In [12]:
column_sampler = np.random.permutation(7)

df.take(column_sampler, axis="columns")  # reorder columns

,4,5,6,0,2,3,1
0,4,5,6,0,2,3,1
1,11,12,13,7,9,10,8
2,18,19,20,14,16,17,15
3,25,26,27,21,23,24,22
4,32,33,34,28,30,31,29


In [13]:
df.sample(n=3)  # 3 random rows without replacement

,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6


In [14]:
# Sampling with replacement — rows can repeat
choices = pd.Series([5, 7, -1, 6, 4])

choices.sample(n=10, replace=True)

1    7
3    6
0    5
3    6
3    6
1    7
2   -1
2   -1
4    4
0    5
dtype: int64

## Computing Indicator / Dummy Variables

Statistical modeling and machine learning often require converting categorical variables into **dummy (indicator) variables** — binary columns, one per category.

### `pd.get_dummies`

For a column with `k` distinct values, produces `k` binary columns:

- `prefix="key"` → adds `key_` prefix to each column name (useful before joining).
- Combine with `df.join(dummies)` to attach the dummies back to the original DataFrame.

### Multiple-Membership Categorical — `str.get_dummies`

When a column encodes **multiple categories in a single string** (e.g. `"Action|Comedy|Drama"`), use `Series.str.get_dummies(sep)`:

- Splits each value on the separator and creates a binary column per unique category.
- Use `dummies.add_prefix("Genre_")` to add a prefix before joining.

### Combining `get_dummies` with `cut`

A useful pattern for modeling is to bin a continuous variable and then one-hot-encode the bins:

```python
pd.get_dummies(pd.cut(values, bins))
```

## Key Points

- `pd.get_dummies(series)` → one binary column per unique value.
- `prefix=` adds a prefix to column names — useful to avoid clashes when joining.
- `series.str.get_dummies(sep)` handles multiple categories encoded in a single string.
- `add_prefix("Genre_")` on a DataFrame adds a prefix to all its column names.
- `pd.get_dummies(pd.cut(...))` one-hot-encodes binned continuous data.

In [15]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"], "data1": range(6)})

df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [16]:
pd.get_dummies(df["key"])

,a,b,c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [17]:
# Add prefix and join back to the original DataFrame
dummies = pd.get_dummies(df["key"], prefix="key")

df[["data1"]].join(dummies)

,data1,key_a,key_b,key_c
0,0,False,True,False
1,1,False,True,False
2,2,True,False,False
3,3,False,False,True
4,4,True,False,False
5,5,False,True,False


In [18]:
# Multiple-membership categorical: genres encoded as "Action|Comedy|Drama"
mnames = ["movie_id", "title", "genres"]

movies = pd.read_table(
    "../datasets/movielens/movies.dat",
    sep="::",
    header=None,
    names=mnames,
    engine="python"
)

movies[:10]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [19]:
# str.get_dummies splits on the separator and creates one column per genre
dummies = movies["genres"].str.get_dummies("|")

dummies.iloc[:10, :6]

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


In [20]:
# Join genre dummies back with a prefix
movies_windic = movies.join(dummies.add_prefix("Genre_"))

movies_windic.iloc[0]

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

In [21]:
# Combine get_dummies with cut: one-hot-encode binned continuous data
np.random.seed(12345)
values = np.random.uniform(size=10)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False
